In [ ]:
import os
import re
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append('/home/sam/Halo_Finder/Final_design/base_script')
from config_io import load_multifield_from_disk
from experiment import build_bg_only_cfg
from bg_stage import run_bg_inference, train_bg_only


def _global_fft_err(x_true, x_hat, n_slices=32):
    """Global FFT magnitude / phase L1 error over a strided set of z-slices."""
    x_true = np.asarray(x_true); x_hat = np.asarray(x_hat)
    D = x_true.shape[0]; step = max(1, D // int(n_slices))
    mag, pha = [], []
    for z in range(0, D, step):
        ft_t = np.fft.rfft2(x_true[z].astype(np.float64), norm="ortho")
        ft_h = np.fft.rfft2(x_hat[z].astype(np.float64), norm="ortho")
        mag.append(float(np.mean(np.abs(np.abs(ft_h) - np.abs(ft_t)))))
        d = np.angle(np.exp(1j * (np.angle(ft_h) - np.angle(ft_t))))
        pha.append(float(np.mean(np.abs(d))))
    return (float(np.mean(mag)) if mag else float("nan"),
            float(np.mean(pha)) if pha else float("nan"))


def _global_diag(x_true, x_hat):
    """Global reconstruction metrics (replaces the old ROI diagnostics)."""
    x_true = np.asarray(x_true); x_hat = np.asarray(x_hat)
    dr = float(x_true.max() - x_true.min()) or 1.0
    mse = float(np.mean((x_true - x_hat) ** 2))
    psnr = 20 * np.log10(dr) - 10 * np.log10(mse + 1e-12) if mse > 0 else 100.0
    max_err = float(np.max(np.abs(x_true - x_hat)))
    fm, fp = _global_fft_err(x_true, x_hat)
    return {"psnr": psnr, "max_err": max_err, "fft_mag_err": fm, "fft_phase_err": fp}

pysz_path = r"/home/sam/Data_Compression/SZ3/tools/pysz"
if pysz_path not in sys.path:
    sys.path.append(pysz_path)
from pysz import SZ


def set_seed(seed=17):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)



In [ ]:
# ==== 路径与数据（可按需改 TARGET_STEM / REL_SETTINGS）====
import sys
from pathlib import Path

base_path = Path(r"/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin").resolve()
base_path = base_path.as_posix() + "/"
sz_lib_path = r"/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
data_shape = (512, 512, 512)

TARGET_STEM = "dark_matter_density"
FIELD_FILES = [
    "dark_matter_density.f32",
    "velocity_z.f32",
    "baryon_density.f32",
    "temperature.f32",
    "velocity_x.f32",
    "velocity_y.f32",
]

REL_SETTINGS = [("r0", 1e-4), ("r1", 2e-4), ("r2", 3e-4),("r3", 4e-4),("r4", 5e-4),("r5", 6e-4),("r6", 1e-5)]
REL_ACTIVE_IDX = 0
REL_ERR_SZ_BITSTREAM = REL_SETTINGS[REL_ACTIVE_IDX][1]


def rel_sz_suffix(rel_err: float) -> str:
    return f"{rel_err:.0e}".replace("+", "")


def sz_bin_for_target(fname: str, rel_err: float) -> str:
    stem = Path(fname).stem
    return base_path + stem + "_rel" + rel_sz_suffix(rel_err) + ".sz"

fname = TARGET_STEM + ".f32"
gt_path = base_path + fname
aux_paths = [base_path + f for f in FIELD_FILES if f != fname]
sz_bin_path = sz_bin_for_target(fname, REL_ERR_SZ_BITSTREAM)

_sz_path = Path(sz_bin_path)
if not _sz_path.is_file():
    from pysz import SZ
    print("[save .sz] 压缩:", _sz_path)
    eng = SZ(sz_lib_path)
    vol = np.fromfile(gt_path, dtype=np.float32).reshape(data_shape)
    blob, cr = eng.compress(vol, 1, 0, REL_ERR_SZ_BITSTREAM, 0)
    del vol
    _sz_path.parent.mkdir(parents=True, exist_ok=True)
    _sz_path.write_bytes(blob)
    print("CR ≈", float(cr))

Xs, Xps = load_multifield_from_disk(
    gt_path=gt_path,
    aux_paths=aux_paths,
    sz_bin_path=sz_bin_path,
    data_shape=data_shape,
    pysz_path=pysz_path,
    sz_lib_path=sz_lib_path,
)
print("Loaded", TARGET_STEM, "| fields", len(Xs))

# (ROI removed -- global metrics only)

sz = SZ(sz_lib_path)
gt_target = np.asarray(Xs[0], np.float32)
aux_fields = [np.asarray(f, np.float32) for f in Xs[1:]]


def build_Xps_for_rel(rel_err: float):
    b, cr = sz.compress(gt_target, 1, 0, float(rel_err), 0)
    x_lq = sz.decompress(b, gt_target.shape, np.float32)
    return [x_lq] + aux_fields, float(cr), b

print("sanity CR:", build_Xps_for_rel(REL_SETTINGS[0][1])[1])



In [ ]:
from pathlib import Path
import re

NEURLZ_CSV_ROOT = Path(
    "/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512"
).resolve()
FIELD_ABBR = {"DMD": "dark_matter_density", "T": "temperature", "BD": "baryon_density"}


def discover_neurlz_csvs(root: Path):
    seen, out = set(), []
    for pattern in ("sz_decompress/*/decompressed_f32/nyx_SF_*.csv", "**/nyx_SF_*.csv"):
        for p in root.glob(pattern):
            if p.is_file() and p.suffix.lower() == ".csv":
                r = p.resolve()
                if r not in seen:
                    seen.add(r)
                    out.append(p)
    return sorted(out, key=lambda p: (str(p.parent), p.name))


def parse_nyx_sf_csv_name(path: Path):
    m = re.match(r"nyx_SF_([A-Za-z0-9]+)_(.+)\.csv$", path.name, re.I)
    return None if not m else (m.group(1).upper(), m.group(2))


def load_neurlz_official_csv(csv_path: Path, *, field_name: str, rel_slug: str, rel_err_float: float) -> dict:
    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]
    for c in ["Epoch", "Average Loss", "PSNR"]:
        if c not in df.columns:
            raise KeyError(c, list(df.columns))
    epochs = df["Epoch"].astype(int).tolist()
    hist = {
        "epoch": epochs,
        "loss": df["Average Loss"].astype(float).tolist(),
        "psnr": list(zip(epochs, df["PSNR"].astype(float).tolist())),
        "psnr_roi": list(zip(epochs, [float("nan")] * len(df))),
        "time": [float("nan")] * len(df),
    }
    metric_log = []
    for _, row in df.iterrows():
        ep = int(row["Epoch"])
        m = {"psnr": float(row["PSNR"]), "epoch": ep}
        if "absDiffMax" in df.columns:
            m["max_err"] = float(row["absDiffMax"])
        metric_log.append(m)
    return {
        "name": f"NeurLZ [{field_name}] rel={rel_slug} [CSV]",
        "cfg": {"bg_arch": "neurlz_unet_official_train_py", "field": field_name, "rel_err": float(rel_err_float)},
        "hist": hist,
        "hint_pretrain_hist": None,
        "metric_log": metric_log,
        "color": "tab:cyan",
        "marker": "X",
        "model": None,
        "rel_err": float(rel_err_float),
        "sz_ratio": float("nan"),
        "sz3_bytes": -1,
        "x_hat": None,
        "ablation_panel": None,
    }

if "results_compare" not in globals():
    results_compare = []

for p in discover_neurlz_csvs(NEURLZ_CSV_ROOT):
    pr = parse_nyx_sf_csv_name(p)
    if not pr:
        continue
    abbr, rel_slug = pr
    field_name = FIELD_ABBR.get(abbr, abbr.lower())
    if field_name != TARGET_STEM:
        continue
    try:
        rf = float(rel_slug)
    except ValueError:
        continue
    results_compare.append(load_neurlz_official_csv(p, field_name=field_name, rel_slug=rel_slug, rel_err_float=rf))

print("NeurLZ CSV entries:", sum(1 for r in results_compare if "[CSV]" in str(r.get("name", ""))))



In [ ]:
import time
import re
from pathlib import Path

set_seed(17)

# 每次 train_bg_variant 结束后保存 checkpoint；改为 None 可关闭落盘
BG_CKPT_ROOT = Path("/home/sam/Halo_Finder/halo_finder_v1/scripts/ablation_ckpts")

BG_TRAIN_TIME = 60.0
BG_LR = 1e-3
BG_PATCH = 512
BG_BATCH = 1
MODEL_DTYPE_BYTES = 2

DEFAULT_BG_ARCH = "micro_film"
DEFAULT_BG_H = 7
DEFAULT_BG_FIELD_NORM = "zscore"
DEFAULT_BG_FREQ_WEIGHT = 0.5
DEFAULT_BG_FFT_PHASE_WEIGHT = 0.5
DEFAULT_BG_FREQ_WARMUP = 1
DEFAULT_BG_SPLIT_MODE = "three"


def model_param_bytes(model, dtype_bytes=None):
    if model is None:
        return 0
    tot = 0
    for p in model.parameters():
        tot += p.numel() * (p.element_size() if dtype_bytes is None else int(dtype_bytes))
    return int(tot)


def psnr_np(x_true, x_hat):
    x_true = np.asarray(x_true, dtype=np.float64)
    x_hat = np.asarray(x_hat, dtype=np.float64)
    mse = float(np.mean((x_true - x_hat) ** 2))
    dr = float(np.max(x_true) - np.min(x_true))
    if mse <= 0:
        return float("inf")
    return float(20.0 * np.log10(max(dr, 1e-12) / np.sqrt(mse)))


def get_run_psnr(r, mode="best"):
    logs = r.get("metric_log", []) or []
    vals = [float(m["psnr"]) for m in logs if m.get("psnr") is not None and np.isfinite(float(m["psnr"]))]
    if vals:
        return float(max(vals) if mode == "best" else vals[-1])
    if r.get("x_hat") is not None:
        return psnr_np(Xs[0], r["x_hat"])
    return float("nan")


def patch_sz3_bytes(r):
    rel = float(r.get("rel_err", float("nan")))
    if not np.isfinite(rel):
        return
    sb = r.get("sz3_bytes", None)
    if sb is None or sb <= 0:
        _, _, b = build_Xps_for_rel(rel)
        r["sz3_bytes"] = int(len(b))
        r["sz_ratio"] = float(gt_target.nbytes / max(len(b), 1))


def _ablation_ckpt_slug(s, max_len=72):
    t = str(s).strip().replace(" ", "_")
    t = re.sub(r"[^\w.\-]+", "_", t)
    return t[:max_len] or "run"


def train_bg_variant(
    tag,
    rel_err,
    *,
    name,
    ablation_panel,
    ablation_tag,
    bg_field_norm=DEFAULT_BG_FIELD_NORM,
    bg_freq_weight=DEFAULT_BG_FREQ_WEIGHT,
    bg_fft_phase_weight=DEFAULT_BG_FFT_PHASE_WEIGHT,
    bg_freq_warmup_epochs=DEFAULT_BG_FREQ_WARMUP,
    bg_split_mode=DEFAULT_BG_SPLIT_MODE,
):
    Xps_rel, sz_ratio, sz_bytes = build_Xps_for_rel(float(rel_err))
    cfg = build_bg_only_cfg(
        X_target=Xs[0],
        Xps=Xps_rel,
        max_train_time=float(BG_TRAIN_TIME),
        bg_h=int(DEFAULT_BG_H),
        roi_h=4,
        epochs=200,
        steps_per_epoch=128,
        bg_patch_size=int(BG_PATCH),
        bg_batch=int(BG_BATCH),
        lr=float(BG_LR),
        bg_field_norm=str(bg_field_norm),
        bg_freq_weight=float(bg_freq_weight),
        bg_fft_phase_weight=float(bg_fft_phase_weight),
        bg_freq_warmup_epochs=int(bg_freq_warmup_epochs),
    )
    cfg.bg_arch = DEFAULT_BG_ARCH
    cfg.bg_split_mode = bg_split_mode
    cfg.bg_split_bands = bg_split_mode in {"two", "three"}
    cfg.bg_split_sigma = 0.12
    cfg.bg_sigma_low = 0.08
    cfg.bg_sigma_mid = 0.18
    cfg.bg_cr_rel_err = float(rel_err)
    cfg.bg_low_weight = 0.25
    cfg.bg_mid_weight = 0.55
    cfg.bg_high_weight = 1.10
    cfg.bg_dyn_band_weight = False
    cfg.bg_band_curriculum = False
    cfg.bg_hard_patch_reweight = False
    cfg.bg_preserve_band_weight_sum = True
    cfg.bg_roi_weight = 0.0
    cfg.bg_random_rel_err = False
    cfg.bg_rel_err_choices = []
    cfg.bg_use_se = bool(globals().get("BG_USE_SE", False))
    cfg.bg_se_reduction = int(globals().get("BG_SE_REDUCTION", 4))

    metric_log = []
    first = [True]

    def evaluator(model):
        if first[0]:
            first[0] = False
            x0 = np.asarray(Xps_rel[0], np.float32)
            m = _global_diag(Xs[0], x0)
            metric_log.append(m)
            return m["psnr"], m["max_err"]
        x_hat = run_bg_inference(model, Xs, Xps_rel, cfg, rel_err)
        m = _global_diag(Xs[0], x_hat)
        metric_log.append(m)
        return m["psnr"], m["max_err"]

    model, hist = train_bg_only(Xs=Xs, Xps=Xps_rel, device=device, cfg=cfg, evaluator=evaluator)

    ckpt_path = None
    if BG_CKPT_ROOT is not None:
        root = Path(BG_CKPT_ROOT)
        root.mkdir(parents=True, exist_ok=True)
        rel_slug = f"{float(rel_err):.0e}".replace("+", "")
        sm = bg_split_mode if bg_split_mode is not None else "none"
        base = (
            f"{_ablation_ckpt_slug(ablation_panel)}__{_ablation_ckpt_slug(ablation_tag)}__"
            f"{_ablation_ckpt_slug(tag)}__rel{rel_slug}__split{_ablation_ckpt_slug(sm)}"
        )
        ckpt_path = root / f"{base}.pt"
        torch.save(
            {
                "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "hist": hist,
                "metric_log": metric_log,
                "meta": {
                    "name": name,
                    "tag": tag,
                    "rel_err": float(rel_err),
                    "ablation_panel": ablation_panel,
                    "ablation_tag": ablation_tag,
                    "bg_arch": getattr(cfg, "bg_arch", None),
                    "bg_field_norm": str(bg_field_norm),
                    "bg_freq_weight": float(bg_freq_weight),
                    "bg_fft_phase_weight": float(bg_fft_phase_weight),
                    "bg_freq_warmup_epochs": int(bg_freq_warmup_epochs),
                    "bg_split_mode": sm,
                    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                    "TARGET_STEM": str(globals().get("TARGET_STEM", "")),
                },
            },
            ckpt_path,
        )
        print(f"[ckpt] saved {ckpt_path}")

    return {
        "name": name,
        "cfg": cfg,
        "hist": hist,
        "hint_pretrain_hist": None,
        "metric_log": metric_log,
        "color": "tab:orange",
        "marker": "o",
        "model": model,
        "rel_err": float(rel_err),
        "sz_ratio": float(sz_ratio),
        "sz3_bytes": int(len(sz_bytes)),
        "ablation_panel": ablation_panel,
        "ablation_tag": ablation_tag,
        "ckpt_path": str(ckpt_path) if ckpt_path is not None else None,
    }



In [ ]:
# --- FFT err vs effective CR：NeurLZ CSV + ablation_ckpts（仅 ladder / M2）---
import re
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from siren_fft_backbone_model import UNET_Model
from experiment import build_bg_only_cfg
from bg_stage import run_bg_inference

SZ_DEC_ROOT = Path(
    "/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/sz_decompress"
)
CKPT_DIR = Path("/home/sam/Halo_Finder/halo_finder_v1/scripts/ablation_ckpts")

FIELD_TO_ABBR = {
    "dark_matter_density": "DMD",
    "temperature": "T",
    "baryon_density": "BD",
}
NEURLZ_NUM_PARAMS = 3235
NEURLZ_DTYPE_BYTES = 4
MODEL_DTYPE_BYTES = int(globals().get("MODEL_DTYPE_BYTES", 2))
INCLUDE_AUX_BYTES = bool(globals().get("INCLUDE_AUX_BYTES", False))
CKPT_BG_H = int(globals().get("DEFAULT_BG_H", 7))

# 只画 ladder 的 M2 这一支
CKPT_PANEL_FILTER = {"ladder"}
CKPT_TAG_SUBSTR = "M2_plus_freq_split3"  # 与 ckpt 文件名 / meta.ablation_tag 一致

_load_kw = {"map_location": "cpu"}
if "weights_only" in inspect.signature(torch.load).parameters:
    _load_kw["weights_only"] = False


def compute_global_fft_errors_np(x_true, x_hat):
    f_true = np.fft.fft2(np.asarray(x_true, np.float64), axes=(-2, -1))
    f_hat = np.fft.fft2(np.asarray(x_hat, np.float64), axes=(-2, -1))
    mag_err = float(np.mean(np.abs(np.abs(f_hat) - np.abs(f_true))))
    phase_diff = np.angle(np.exp(1j * (np.angle(f_hat) - np.angle(f_true))))
    phase_err = float(np.mean(np.abs(phase_diff)))
    return mag_err, phase_err


def patch_sz3_bytes_row(rel_err: float) -> float:
    _, _, b = build_Xps_for_rel(float(rel_err))
    return float(len(b))


def effective_cr_from_bytes(rel_err: float, nn_param_bytes: float) -> float:
    x_true = np.asarray(Xs[0], np.float32)
    orig = int(x_true.nbytes)
    aux_b = 0
    if INCLUDE_AUX_BYTES and len(Xs) > 1:
        aux_b = int(sum(np.asarray(f, np.float32).nbytes for f in Xs[1:]))
    sz_b = patch_sz3_bytes_row(rel_err)
    tot = sz_b + float(nn_param_bytes) + aux_b
    return float(orig / tot)


def build_cfg_like_train_bg_variant(Xps_rel, rel_err: float, meta: dict):
    sm_raw = meta.get("bg_split_mode")
    if sm_raw is None or str(sm_raw).lower() == "none":
        bg_split_mode = None
    else:
        bg_split_mode = str(sm_raw)

    cfg = build_bg_only_cfg(
        X_target=Xs[0],
        Xps=Xps_rel,
        max_train_time=1.0,
        bg_h=int(CKPT_BG_H),
        roi_h=4,
        epochs=1,
        steps_per_epoch=1,
        bg_patch_size=int(globals().get("BG_PATCH", 512)),
        bg_batch=1,
        lr=1e-3,
        bg_field_norm=str(meta.get("bg_field_norm", "zscore")),
        bg_freq_weight=float(meta.get("bg_freq_weight", 0.0)),
        bg_fft_phase_weight=float(meta.get("bg_fft_phase_weight", 0.0)),
        bg_freq_warmup_epochs=int(meta.get("bg_freq_warmup_epochs", 1)),
    )
    cfg.bg_arch = str(meta.get("bg_arch", globals().get("DEFAULT_BG_ARCH", "micro_film")))
    cfg.bg_split_mode = bg_split_mode
    cfg.bg_split_bands = bg_split_mode in {"two", "three"}
    cfg.bg_split_sigma = 0.12
    cfg.bg_sigma_low = 0.08
    cfg.bg_sigma_mid = 0.18
    cfg.bg_cr_rel_err = float(rel_err)
    cfg.bg_low_weight = 0.25
    cfg.bg_mid_weight = 0.55
    cfg.bg_high_weight = 1.10
    cfg.bg_dyn_band_weight = False
    cfg.bg_band_curriculum = False
    cfg.bg_hard_patch_reweight = False
    cfg.bg_preserve_band_weight_sum = True
    cfg.bg_roi_weight = 0.0
    cfg.bg_random_rel_err = False
    cfg.bg_rel_err_choices = []
    cfg.bg_use_se = bool(globals().get("BG_USE_SE", False))
    cfg.bg_se_reduction = int(globals().get("BG_SE_REDUCTION", 4))
    return cfg


def load_model_from_ckpt(ckpt: dict, meta: dict):
    D, H, W = (int(x) for x in Xs[0].shape)
    sm = meta.get("bg_split_mode")
    if sm is None or str(sm).lower() == "none":
        sm = None
    else:
        sm = str(sm)
    model = UNET_Model(
        n_fields=len(Xs),
        K=7,
        D=D,
        H=H,
        W=W,
        bg_hidden=int(CKPT_BG_H),
        bg_arch=str(meta.get("bg_arch", DEFAULT_BG_ARCH)),
        bg_split_bands=sm in {"two", "three"},
        bg_split_mode=sm,
        bg_use_se=bool(globals().get("BG_USE_SE", False)),
        bg_se_reduction=int(globals().get("BG_SE_REDUCTION", 4)),
    ).to(device)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    return model


def nn_bytes_cascaded(model) -> int:
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(n) * int(MODEL_DTYPE_BYTES)


def want_ckpt(pt_path: Path, meta: dict) -> bool:
    if str(meta.get("TARGET_STEM", TARGET_STEM)) != str(TARGET_STEM):
        return False
    panel = meta.get("ablation_panel")
    if CKPT_PANEL_FILTER is not None and panel not in CKPT_PANEL_FILTER:
        return False
    tag = str(meta.get("ablation_tag", ""))
    if CKPT_TAG_SUBSTR not in tag and CKPT_TAG_SUBSTR not in pt_path.name:
        return False
    return True


# --------- 1) NeurLZ CSV ---------
rows = []
abbr = FIELD_TO_ABBR.get(TARGET_STEM)
if not abbr:
    raise KeyError(f"在 FIELD_TO_ABBR 中为 {TARGET_STEM} 增加缩写")

pat = re.compile(rf"nyx_SF_{abbr}_(.+)\.csv$", re.I)
for csv_path in sorted(SZ_DEC_ROOT.rglob("nyx_SF_*.csv")):
    m = pat.match(csv_path.name)
    if not m:
        continue
    rel_slug = m.group(1)
    try:
        rel_err = float(rel_slug.replace("E", "e"))
    except ValueError:
        continue

    df_csv = pd.read_csv(csv_path)
    for c in ["PSNR", "fft_mag_err", "fft_phase_err"]:
        if c not in df_csv.columns:
            raise KeyError(f"{csv_path} 缺少列 {c}")

    i_best = int(df_csv["PSNR"].astype(float).idxmax())
    row = df_csv.iloc[i_best]
    nn_b = float(NEURLZ_NUM_PARAMS * NEURLZ_DTYPE_BYTES)
    cr = effective_cr_from_bytes(rel_err, nn_b)

    rows.append({
        "source": "NeurLZ_CSV",
        "label": "NeurLZ (CSV best-epoch)",
        "rel_err": rel_err,
        "effective_cr": cr,
        "fft_mag_err": float(row["fft_mag_err"]),
        "fft_phase_err": float(row["fft_phase_err"]),
        "psnr": float(row["PSNR"]),
        "path": str(csv_path),
    })

# --------- 2) ablation_ckpts：仅 ladder M2 ---------
for pt_path in sorted(CKPT_DIR.glob("*.pt")):
    ckpt = torch.load(pt_path, **_load_kw)
    meta = ckpt.get("meta") or {}
    if not want_ckpt(pt_path, meta):
        continue

    rel_err = float(meta.get("rel_err"))
    Xps_rel, _, _ = build_Xps_for_rel(rel_err)
    cfg = build_cfg_like_train_bg_variant(Xps_rel, rel_err, meta)

    try:
        model = load_model_from_ckpt(ckpt, meta)
    except Exception as e:
        print(f"[skip model] {pt_path.name}: {e}")
        continue

    x_hat = run_bg_inference(model, Xs, Xps_rel, cfg, rel_err)
    mag_e, ph_e = compute_global_fft_errors_np(Xs[0], x_hat)
    nn_b = float(nn_bytes_cascaded(model))
    cr = effective_cr_from_bytes(rel_err, nn_b)
    tag = str(meta.get("ablation_tag", CKPT_TAG_SUBSTR))
    rows.append({
        "source": "ckpt",
        "label": f"ladder|{tag}",  # 单 pipeline：图上就一条 ckpt 曲线
        "rel_err": rel_err,
        "effective_cr": cr,
        "fft_mag_err": mag_e,
        "fft_phase_err": ph_e,
        "psnr": float(20.0 * np.log10(max(float(np.ptp(Xs[0])), 1e-12) / max(float(np.sqrt(np.mean((Xs[0] - x_hat) ** 2))), 1e-12))),
        "path": str(pt_path),
    })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --------- 3) 表 + 图 ---------
df_fft = pd.DataFrame(rows)
if df_fft.empty:
    raise RuntimeError("没有数据：检查 TARGET_STEM / ckpt meta / 文件名是否含 M2_plus_freq_split3")

display(df_fft.sort_values(["label", "rel_err"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=140)
cmap = plt.get_cmap("tab10")
labels = sorted(df_fft["label"].unique())
for i, lab in enumerate(labels):
    g = df_fft[df_fft["label"] == lab].sort_values("effective_cr")
    c = cmap(i % 10)
    ax1.plot(g["effective_cr"], g["fft_mag_err"], marker="o", linewidth=2, color=c, label=lab)
    ax2.plot(g["effective_cr"], g["fft_phase_err"], marker="o", linewidth=2, color=c, label=lab)

ax1.set_xlabel("Effective CR = original / (SZ3 + NN params)")
ax1.set_ylabel("Global FFT mag err")
ax1.set_title(f"FFT magnitude | {TARGET_STEM}")
ax1.grid(True, alpha=0.35)
ax2.set_xlabel("Effective CR = original / (SZ3 + NN params)")
ax2.set_ylabel("Global FFT phase err")
ax2.set_title(f"FFT phase | {TARGET_STEM}")
ax2.grid(True, alpha=0.35)
ax2.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()